**Гипотеза:** расписание описывает ту же сеть рейсов, но на год позже.

In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
BLUE, ORANGE, INK, GRID = '#2a78d6', '#eb6834', '#52514e', '#e6e5e0'

con = duckdb.connect()
con.sql("create table s as select * from 'dim_flight_schedule.parquet' where source_file like 'Skyteam%'")
con.sql("""create table t as
    select trip_id, flight_number fn, flight_date dt, origin_code o, dest_code d, schedule_match, data_quality_flags
    from 'output/fact_trips.parquet'
    where schedule_match is distinct from 'exact_dates' and origin_code is not null""")
con.sql("select min(valid_from) расписание_с, max(valid_to) расписание_по, count(*) строк, count(distinct flight_number) рейсов from s").df()

,расписание_с,расписание_по,строк,рейсов
0,2018-11-01,2019-01-31,55469,15354


In [2]:
variants = {
    'сдвиг +1 год': "t.dt + interval 1 year between s.valid_from and s.valid_to",
    'сдвиг +1 год и 45 дней': "t.dt + interval 1 year + interval 45 day between s.valid_from and s.valid_to",
    'сдвиг +2 года': "t.dt + interval 2 year between s.valid_from and s.valid_to",
    'без сдвига': "t.dt between s.valid_from and s.valid_to",
    'только номер рейса, даты не учитываются': "true",
    'период расписания НЕ покрывает дату +1 год': "not (t.dt + interval 1 year between s.valid_from and s.valid_to)",
}
rows = []
for name, cond in variants.items():
    r = con.sql(f"""select count(*) n, avg(ok::int) * 100 pct from (
        select t.trip_id, bool_or(s.origin_code = t.o and s.dest_code = t.d) ok
        from t join s on s.flight_number = t.fn and {cond} group by 1)""").fetchone()
    rows.append({'вариант сопоставления': name, 'перелётов нашлось в расписании': r[0],
                 'маршрут совпал, %': None if r[1] is None else round(r[1], 1)})
V = pd.DataFrame(rows)
V

,вариант сопоставления,перелётов нашлось в расписании,"маршрут совпал, %"
0,сдвиг +1 год,156607,67.2
1,сдвиг +1 год и 45 дней,218629,67.4
2,сдвиг +2 года,44819,91.4
3,без сдвига,0,NaN
4,"только номер рейса, даты не учитываются",1099454,70.6
5,период расписания НЕ покрывает дату +1 год,1009775,68.5


In [3]:
M = con.sql("""
    with base as (
        select month(t.dt) m, t.trip_id, bool_or(s.origin_code = t.o and s.dest_code = t.d) ok
        from t join s on s.flight_number = t.fn group by 1, 2),
    sh as (
        select month(t.dt) m, t.trip_id, bool_or(s.origin_code = t.o and s.dest_code = t.d) ok
        from t join s on s.flight_number = t.fn and t.dt + interval 1 year between s.valid_from and s.valid_to
        where year(t.dt) = 2017 group by 1, 2)
    select b.m месяц, round(avg(b.ok::int) * 100, 1) только_номер_рейса,
           round((select avg(ok::int) * 100 from sh where sh.m = b.m), 1) сдвиг_1_год,
           (select count(*) from sh where sh.m = b.m) найдено_со_сдвигом
    from base b group by 1 order by 1""").df()
M

,месяц,только_номер_рейса,сдвиг_1_год,найдено_со_сдвигом
0,1,89.8,NaN,0
1,2,86.8,NaN,0
2,3,81.7,NaN,0
3,4,77.6,NaN,0
4,5,74.1,NaN,0
5,6,71.1,NaN,0
6,7,68.6,NaN,0
7,8,66.3,NaN,0
8,9,64.7,NaN,0
9,10,63.1,NaN,0


In [5]:
con.sql("""
    select coalesce(schedule_match, 'не найдено') сопоставление, count(*) перелётов,
           count(sched_dep_time) плановое_время, count(travel_time_min) время_в_пути, count(aircraft_code) тип_самолёта,
           count(*) filter (where data_quality_flags like '%iata_from_schedule%') аэропорт_восстановлен
    from 'output/fact_trips.parquet' group by 1 order by 2""").df()

,сопоставление,перелётов,плановое_время,время_в_пути,тип_самолёта,аэропорт_восстановлен
0,year_shift_1,105175,105175,105175,105174,1518
1,exact_dates,243788,243788,240189,0,0
2,не найдено,1649726,0,0,0,0


In [6]:
con.sql("""
    select origin_code || ' → ' || dest_code маршрут, count(*) перелётов
    from 'output/fact_trips.parquet' where data_quality_flags like '%iata_from_schedule%'
    group by 1 order by 2 desc limit 10""").df()

,маршрут,перелётов
0,GMP → CJU,340
1,CJU → PUS,65
2,CJU → CJJ,50
3,USN → GMP,49
4,GMP → USN,43
5,TAE → CJU,37
6,CJJ → CJU,32
7,NBO → ZNZ,31
8,PUS → CJU,29
9,TPE → PUS,28
